# P5e 探索と採用構成の再検査
固定コミット未設定なら停止。CUDAエラー後はランタイムを再起動し最初から実行。探索の計時は同一セッションで完走すること。


In [ ]:
CODE_COMMIT = "562bd48e6ede061fbaa4621c3719b5223fc24dca"
from pathlib import Path
import re, subprocess, sys, os
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("固定コミット未設定。停止します")
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/viveMonte-p5e')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/gakugaku3333/chatcarlo.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() != CODE_COMMIT:
    raise ValueError('コミット不一致')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from prototypes.mv_electron.gpu import p5e_colab, p5e_core
BUNDLE = Path('/content/drive/MyDrive/viveMonte/p5/p5b')
RESULTS = Path('/content/drive/MyDrive/viveMonte/p5/p5e')
EXPECTED_INPUT_SHA256 = '7b02611de118c11add0471744e64fc1b77cfb0aeb4fe285e4f29b9e7699d920a'
EXPECTED_REFERENCE_SHA256 = '02f01237752230937def6c9ffdecefbbab398d9d600a5ec890b7ad16a08b289f'
EXPECTED_SOURCE_SHA256 = '2ce5fe575ee6317b1be5cd3c0f319ed1cd9bc07b19fb104e1addcb11782832d0'
assert p5e_core.d.BUNDLE_SHA == {'p5b_inputs.npz': EXPECTED_INPUT_SHA256, 'p5b_reference.npz': EXPECTED_REFERENCE_SHA256}
assert p5e_core.d.SOURCE_SHA == EXPECTED_SOURCE_SHA256
p5e_core.d.load_bundle(BUNDLE)


In [ ]:
report = p5e_colab.explore(BUNDLE, RESULTS/'explore')
report


In [ ]:
ADOPT_VARIANT = ''  # 手元で再判定した候補だけを指定
if ADOPT_VARIANT not in p5e_core.VARIANTS or ADOPT_VARIANT == 'V0':
    raise ValueError('採用候補未設定。停止します')
p5e_colab.adopt(BUNDLE, RESULTS/'adopt', ADOPT_VARIANT)
